In [13]:
from pathlib import Path

home = Path.home()
search_dirs = [
    home / "Downloads",
    home / "Documents",
    home / "datasets",
    home / "data",
    home / "Desktop",
]

keywords = ("scannet", "scannetv2", "scannet_v2")

print("=== ScanNetV2 folders and files ===")
found = []

for base in search_dirs:
    if not base.exists():
        continue

    for p in base.rglob("*"):
        if any(k in p.name.lower() for k in keywords):
            found.append(p)
            if len(found) >= 100:
                break

    if len(found) >= 100:
        break

if found:
    for p in found:
        print(("DIR " if p.is_dir() else "FILE"), p)
else:
    print("No ScanNetV2-named paths found in the searched locations.")

print("\n=== Common test-set indicators ===")
for base in search_dirs:
    if base.exists():
        for name in ("scannetv2_test.txt", "scannetv2_test", "test"):
            p = base / name
            if p.exists():
                print(p)

=== ScanNetV2 folders and files ===
No ScanNetV2-named paths found in the searched locations.

=== Common test-set indicators ===


In [14]:

from pathlib import Path
import os
import sys
import torch

print("=" * 60)
print("SCR-CMX ENVIRONMENT CHECK")
print("=" * 60)

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

search_roots = [
    Path.home() / "Downloads",
    Path.home() / "projects",
    Path.home() / "Documents",
    Path.cwd(),
]

print("\nSearching for CMX repositories...")

found = set()

for root in search_roots:
    if not root.exists():
        continue

    if root.name == "CMX_Original":
        found.add(root.resolve())

    try:
        for p in root.glob("**/CMX_Original"):
            if p.is_dir():
                found.add(p.resolve())
    except (PermissionError, OSError):
        pass

if found:
    for repo in sorted(found):
        print("\nRepository:", repo)
        for name in ["builder.py", "train.py", "eval.py", "config.py"]:
            matches = list(repo.rglob(name))
            if matches:
                print(f"  {name}: {matches[0]}")
else:
    print("CMX_Original not found in common locations.")
    print("Current notebook directory:", Path.cwd())
    print("Please set the repository path manually.")


SCR-CMX ENVIRONMENT CHECK
Python: 3.10.21 (main, Aug 27 2026, 14:42:07) [GCC 14.3.0]
PyTorch: 2.14.1+cu130
CUDA available: True
GPU: NVIDIA RTX 2000 Ada Generation
GPU memory: 15.57 GB

Searching for CMX repositories...

Repository: /home/cse-sdpl/Downloads/CMX_Original
  builder.py: /home/cse-sdpl/Downloads/CMX_Original/models/builder.py
  train.py: /home/cse-sdpl/Downloads/CMX_Original/train.py
  eval.py: /home/cse-sdpl/Downloads/CMX_Original/eval.py
  config.py: /home/cse-sdpl/Downloads/CMX_Original/config.py


In [15]:

from pathlib import Path
import re

ROOT = Path("/home/cse-sdpl/Downloads")

# Find the existing repositories without modifying anything
candidates = [
    ROOT / "CMX_Original",
    ROOT / "CMX_E3",
    ROOT / "CMX_E4",
    ROOT / "CMX_E5",
]

repos = [p for p in candidates if p.is_dir()]

print("=" * 75)
print("SCR-CMX: EXISTING ARCHITECTURE INSPECTION")
print("=" * 75)

for repo in repos:
    print(f"\nREPOSITORY: {repo}")

    files_to_check = [
        repo / "models" / "builder.py",
        repo / "models" / "net_utils.py",
        repo / "train.py",
        repo / "config.py",
    ]

    for file_path in files_to_check:
        if not file_path.exists():
            continue

        print(f"\n--- {file_path.relative_to(repo)} ---")

        # Print architecture declarations and relevant implementation lines.
        lines = file_path.read_text(errors="replace").splitlines()
        keywords = re.compile(
            r"class |def |FRM|FFM|reliab|counterfactual|"
            r"fusion|rectif|modality_logits|mit_b2_e[345]|"
            r"backbone|forward\("
        )

        matches = [
            (i, line) for i, line in enumerate(lines)
            if keywords.search(line)
        ]

        if not matches:
            print("(No matching declarations found.)")
            continue

        # Limit output to keep the notebook manageable.
        for i, line in matches[:100]:
            print(f"{i + 1:4}: {line[:180]}")

        if len(matches) > 100:
            print(f"... {len(matches) - 100} additional matching lines omitted.")

print("\n" + "=" * 75)
print("INSPECTION COMPLETE — NO FILES MODIFIED")
print("=" * 75)


SCR-CMX: EXISTING ARCHITECTURE INSPECTION

REPOSITORY: /home/cse-sdpl/Downloads/CMX_Original

--- models/builder.py ---
  13: class EncoderDecoder(nn.Module):
  14:     def __init__(self, cfg=None, criterion=nn.CrossEntropyLoss(reduction='mean', ignore_index=255), norm_layer=nn.BatchNorm2d):
  18:         # import backbone and decoder
  19:         if cfg.backbone == 'swin_s':
  20:             logger.info('Using backbone: Swin-Transformer-small')
  21:             from .encoders.dual_swin import swin_s as backbone
  23:             self.backbone = backbone(norm_fuse=norm_layer)
  24:         elif cfg.backbone == 'swin_b':
  25:             logger.info('Using backbone: Swin-Transformer-Base')
  26:             from .encoders.dual_swin import swin_b as backbone
  28:             self.backbone = backbone(norm_fuse=norm_layer)
  29:         elif cfg.backbone == 'mit_b5':
  30:             logger.info('Using backbone: Segformer-B5')
  31:             from .encoders.dual_segformer import mi

In [16]:

from pathlib import Path

BASE = Path("/home/cse-sdpl/Downloads")

files = {
    "E3 encoder": BASE / "CMX_E3/models/encoders/dual_segformer_e3.py",
    "E4 encoder": BASE / "CMX_E4/models/encoders/dual_segformer_e4.py",
    "E5 encoder": BASE / "CMX_E5/models/encoders/dual_segformer_e5.py",
    "E5 builder": BASE / "CMX_E5/models/builder.py",
}

for name, path in files.items():
    print("\n" + "=" * 90)
    print(name, "-", path)
    print("=" * 90)

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    lines = path.read_text(errors="replace").splitlines()

    # Print architecture setup and forward methods.
    ranges = []
    for i, line in enumerate(lines):
        if (
            line.lstrip().startswith("class ")
            or line.lstrip().startswith("def forward")
            or "FRM" in line
            or "FFM" in line
            or "reliability" in line.lower()
            or "counterfactual" in line.lower()
        ):
            ranges.append(i)

    selected = set()
    for i in ranges:
        selected.update(range(max(0, i - 3), min(len(lines), i + 5)))

    previous = -2
    for i in sorted(selected):
        if i > previous + 1:
            print("    ...")
        print(f"{i+1:4}: {lines[i]}")
        previous = i

print("\nInspection finished. No files were modified.")



E3 encoder - /home/cse-sdpl/Downloads/CMX_E3/models/encoders/dual_segformer_e3.py
    ...
   4: from functools import partial
   5: 
   6: from timm.models.layers import DropPath, to_2tuple, trunc_normal_
   7: from ..net_utils_e3 import ReliabilityGuidedFusionModule as FFM
   8: from ..net_utils import FeatureRectifyModule as FRM
   9: import math
  10: import time
  11: from engine.logger import get_logger
  12: 
  13: logger = get_logger()
  14: 
  15: 
  16: class DWConv(nn.Module):
  17:     """
  18:     Depthwise convolution bloc: input: x with size(B N C); output size (B N C)
  19:     """
  20:     def __init__(self, dim=768):
  21:         super(DWConv, self).__init__()
  22:         self.dwconv = nn.Conv2d(dim, dim, kernel_size=3, stride=1, padding=1, bias=True, groups=dim)
  23: 
  24:     def forward(self, x, H, W):
  25:         B, N, C = x.shape
  26:         x = x.permute(0, 2, 1).reshape(B, C, H, W).contiguous() # B N C -> B C N -> B C H W
  27:         x = self.dwcon

In [17]:
from pathlib import Path

base = Path("/home/cse-sdpl/Downloads")
files_to_inspect = [
    base / "CMX_E3/models/net_utils_e3.py",
    base / "CMX_E4/models/net_utils_e4.py",
    base / "CMX_E5/models/net_utils_e5.py",
    base / "CMX_Original/models/net_utils.py",
]

keywords = (
    "class Reliability",
    "class FeatureRectify",
    "class FeatureFusion",
    "def forward",
    "reliability",
    "gate",
    "sigmoid",
    "softmax",
)

for path in files_to_inspect:
    print("\n" + "=" * 100)
    print(path)
    print("=" * 100)

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    lines = path.read_text(errors="replace").splitlines()
    selected = set()

    for i, line in enumerate(lines):
        if any(word.lower() in line.lower() for word in keywords):
            selected.update(range(max(0, i - 4), min(len(lines), i + 9)))

    previous = -2
    for i in sorted(selected):
        if i > previous + 1:
            print("...")
        print(f"{i + 1:4}: {lines[i]}")
        previous = i


/home/cse-sdpl/Downloads/CMX_E3/models/net_utils_e3.py
...
   3: 
   4: from .net_utils import FeatureFusionModule
   5: 
   6: 
   7: class ReliabilityGuidedFusionModule(FeatureFusionModule):
   8:     """
   9:     E3 prototype:
  10:     Preserve CMX CrossPath and ChannelEmbed, but estimate spatially
  11:     varying RGB/thermal fusion weights after cross-modal interaction.
  12:     """
  13: 
  14:     def __init__(self, dim, reduction=1, num_heads=None,
  15:                  norm_layer=nn.BatchNorm2d):
...
  20:             norm_layer=norm_layer,
  21:         )
  22: 
  23:         hidden = max(dim // 8, 8)
  24:         self.reliability_head = nn.Sequential(
  25:             nn.Conv2d(dim * 2, hidden, kernel_size=1),
  26:             nn.ReLU(inplace=True),
  27:             nn.Conv2d(hidden, 2, kernel_size=1),
  28:         )
  29: 
  30:     def forward(self, x1, x2):
  31:         B, C, H, W = x1.shape
  32: 
  33:         # Original CMX cross-modal interaction.
  34:   

In [18]:

from pathlib import Path

base = Path("/home/cse-sdpl/Downloads/CMX_E5")

targets = {
    "E5 builder": base / "models/builder.py",
    "E5 training": base / "train.py",
    "E5 reliability module": base / "models/net_utils_e5.py",
    "E5 encoder": base / "models/encoders/dual_segformer_e5.py",
}

search_terms = (
    "counterfactual",
    "reliability_loss",
    "reliability_maps",
    "last_reliabilities",
    "e5_corruption_fraction",
    "e5_counterfactual_temperature",
    "loss_seg",
    "loss_total",
)

for name, path in targets.items():
    print("\n" + "=" * 100)
    print(name, "-", path)
    print("=" * 100)

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    lines = path.read_text(errors="replace").splitlines()
    hits = [
        i for i, line in enumerate(lines)
        if any(term.lower() in line.lower() for term in search_terms)
    ]

    selected = set()
    for i in hits:
        selected.update(range(max(0, i - 10), min(len(lines), i + 18)))

    previous = -2
    for i in sorted(selected):
        if i > previous + 1:
            print("...")
        print(f"{i + 1:4}: {lines[i]}")
        previous = i



E5 builder - /home/cse-sdpl/Downloads/CMX_E5/models/builder.py
...
  36:             self.backbone = backbone(norm_fuse=norm_layer)
  37:         elif cfg.backbone == 'mit_b2_e3':
  38:             logger.info('Using E3: SegFormer-B2 with reliability-guided fusion')
  39:             from .encoders.dual_segformer_e3 import mit_b2 as backbone
  40:             self.backbone = backbone(norm_fuse=norm_layer)
  41:         elif cfg.backbone == 'mit_b2_e4':
  42:             logger.info('Using E4: SegFormer-B2 with reliability-gated rectification')
  43:             from .encoders.dual_segformer_e4 import mit_b2 as backbone
  44:             self.backbone = backbone(norm_fuse=norm_layer)
  45:         elif cfg.backbone == 'mit_b2_e5':
  46:             logger.info('Using E5: SegFormer-B2 with counterfactual reliability')
  47:             from .encoders.dual_segformer_e5 import mit_b2 as backbone
  48:             self.backbone = backbone(norm_fuse=norm_layer)
  49:         elif cfg.backbo

In [19]:

from pathlib import Path

files = {
    "E5 counterfactual loss": (
        Path("/home/cse-sdpl/Downloads/CMX_E5/models/builder.py"),
        150, 335,
    ),
    "E5 reliability module": (
        Path("/home/cse-sdpl/Downloads/CMX_E5/models/net_utils_e5.py"),
        1, 100,
    ),
    "E3 fusion module": (
        Path("/home/cse-sdpl/Downloads/CMX_E3/models/net_utils_e3.py"),
        1, 100,
    ),
}

for title, (path, start, end) in files.items():
    print("\n" + "=" * 100)
    print(title)
    print(path)
    print("=" * 100)

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    lines = path.read_text(errors="replace").splitlines()
    for i in range(start - 1, min(end, len(lines))):
        print(f"{i + 1:4}: {lines[i]}")



E5 counterfactual loss
/home/cse-sdpl/Downloads/CMX_E5/models/builder.py
 150:     def _e5_corrupt_modality(self, image):
 151:         """Mask one random rectangle per image after normalization."""
 152:         import math
 153: 
 154:         batch, channels, height, width = image.shape
 155:         fraction = min(max(self.e5_corruption_fraction, 0.01), 0.90)
 156:         block_h = max(1, min(height, int(height * math.sqrt(fraction))))
 157:         block_w = max(1, min(width, int(width * math.sqrt(fraction))))
 158: 
 159:         corrupted = image.clone()
 160:         for b in range(batch):
 161:             y0 = int(torch.randint(
 162:                 0, height - block_h + 1, (1,), device=image.device
 163:             ).item())
 164:             x0 = int(torch.randint(
 165:                 0, width - block_w + 1, (1,), device=image.device
 166:             ).item())
 167:             corrupted[b, :, y0:y0 + block_h, x0:x0 + block_w] = 0
 168: 
 169:         return corrupte

In [20]:

from pathlib import Path
import shutil
import subprocess
import torch

# Existing experiment — read-only source
E5_DIR = Path("/home/cse-sdpl/Downloads/CMX_E5")

# New experiment — separate working copy
SCR_DIR = Path("/home/cse-sdpl/Downloads/CMX_SCR")

print("=" * 70)
print("SCR-CMX INITIAL SAFETY CHECK")
print("=" * 70)

# 1. Verify E5 source
if not E5_DIR.exists():
    raise FileNotFoundError(f"E5 directory not found: {E5_DIR}")

print(f"\nE5 source exists: {E5_DIR}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

# 2. Inspect important source files
important_files = [
    "train.py",
    "eval.py",
    "config.py",
    "models/builder.py",
    "models/net_utils_e5.py",
    "models/encoders/dual_segformer_e5.py",
]

print("\n--- Important E5 files ---")
for relative_path in important_files:
    path = E5_DIR / relative_path
    print(f"{'FOUND' if path.exists() else 'MISSING'}: {relative_path}")

# 3. Check E5 checkpoints without modifying them
checkpoint_dir = E5_DIR / "log_TDD_mit_b2_e5" / "checkpoint"
print("\n--- E5 checkpoints ---")
if checkpoint_dir.exists():
    for checkpoint in sorted(checkpoint_dir.glob("*.pth")):
        print(f"{checkpoint.name}: {checkpoint.stat().st_size / 1024**2:.2f} MB")
else:
    print("Checkpoint directory not found at the expected location.")

# 4. Safely create a separate workspace by copying E5
if SCR_DIR.exists():
    print(f"\nSCR workspace already exists: {SCR_DIR}")
    print("Keeping it unchanged. No overwrite performed.")
else:
    print("\nCreating separate SCR workspace from E5...")
    shutil.copytree(
        E5_DIR,
        SCR_DIR,
        ignore=shutil.ignore_patterns(
            "__pycache__", "*.pyc", ".ipynb_checkpoints"
        )
    )
    print(f"Created: {SCR_DIR}")

# 5. Confirm both directories remain separate
print("\n--- Safety summary ---")
print(f"E5 source: {E5_DIR}")
print(f"SCR workspace: {SCR_DIR}")
print(f"Same directory: {E5_DIR.resolve() == SCR_DIR.resolve()}")
print("\nInitial inspection complete. No training started.")


SCR-CMX INITIAL SAFETY CHECK

E5 source exists: /home/cse-sdpl/Downloads/CMX_E5
PyTorch version: 2.14.1+cu130
CUDA available: True
GPU: NVIDIA RTX 2000 Ada Generation
GPU memory: 15.57 GB

--- Important E5 files ---
FOUND: train.py
FOUND: eval.py
FOUND: config.py
FOUND: models/builder.py
FOUND: models/net_utils_e5.py
FOUND: models/encoders/dual_segformer_e5.py

--- E5 checkpoints ---
epoch-10.pth: 764.97 MB
epoch-100.pth: 764.97 MB
epoch-15.pth: 764.97 MB
epoch-20.pth: 764.97 MB
epoch-25.pth: 764.97 MB
epoch-30.pth: 764.97 MB
epoch-35.pth: 764.97 MB
epoch-40.pth: 764.97 MB
epoch-45.pth: 764.97 MB
epoch-5.pth: 764.96 MB
epoch-50.pth: 764.97 MB
epoch-55.pth: 764.97 MB
epoch-60.pth: 764.97 MB
epoch-65.pth: 764.97 MB
epoch-70.pth: 764.97 MB
epoch-75.pth: 764.97 MB
epoch-80.pth: 764.97 MB
epoch-85.pth: 764.97 MB
epoch-90.pth: 764.97 MB
epoch-95.pth: 764.97 MB
epoch-last.pth: 764.97 MB

Creating separate SCR workspace from E5...
Created: /home/cse-sdpl/Downloads/CMX_SCR

--- Safety summary -

In [ ]:

from pathlib import Path
import shutil
import re

SCR_DIR = Path("/home/cse-sdpl/Downloads/CMX_SCR")

print("=" * 75)
print("SCR-CMX: DISK AND ARCHITECTURE INSPECTION")
print("=" * 75)

# 1. Disk space check
usage = shutil.disk_usage(SCR_DIR)
print("\n--- Disk usage ---")
print(f"Total:     {usage.total / 1024**3:.2f} GB")
print(f"Used:      {usage.used / 1024**3:.2f} GB")
print(f"Free:      {usage.free / 1024**3:.2f} GB")

if usage.free / 1024**3 < 20:
    print("WARNING: Less than 20 GB free. Avoid another full checkpoint copy.")

# 2. Check important files in the SCR copy
files_to_inspect = [
    "models/net_utils_e5.py",
    "models/encoders/dual_segformer_e5.py",
    "models/builder.py",
    "train.py",
    "config.py",
]

for relative_path in files_to_inspect:
    path = SCR_DIR / relative_path
    print(f"\n{'=' * 75}\nFILE: {relative_path}")
    if not path.exists():
        print("FILE MISSING")
        continue

    lines = path.read_text(errors="replace").splitlines()
    print(f"Total lines: {len(lines)}")

    # Locate architecture and loss-related definitions
    patterns = re.compile(
        r"^\s*(class |def |    def |    class )|"
        r"reliability|counterfactual|FeatureFusion|FeatureRectify|"
        r"backbone|optimizer|criterion|loss"
    )

    matches = [
        (i, line) for i, line in enumerate(lines)
        if patterns.search(line)
    ]

    # Print matches with nearby context, merging overlapping ranges
    ranges = []
    for i, _ in matches:
        start, end = max(0, i - 2), min(len(lines), i + 4)
        if ranges and start <= ranges[-1][1]:
            ranges[-1] = (ranges[-1][0], max(ranges[-1][1], end))
        else:
            ranges.append((start, end))

    # Limit output so the notebook does not become enormous
    printed = 0
    max_lines = 240 if "net_utils" in relative_path else 200

    for start, end in ranges:
        chunk_len = end - start
        if printed + chunk_len > max_lines:
            print("... output limit reached; remaining sections omitted ...")
            break

        print(f"\n--- Lines {start + 1}-{end} ---")
        for j in range(start, end):
            print(f"{j + 1:4}: {lines[j]}")
        printed += chunk_len

print("\nInspection complete. No source files were modified.")


In [21]:

from pathlib import Path
import difflib

ROOT = Path("/home/cse-sdpl/Downloads/CMX_SCR")

files_to_check = [
    (
        "config.py",
        "config.py.before_scr_identity",
    ),
    (
        "models/builder.py",
        "models/builder.py.before_scr_identity",
    ),
    (
        "models/encoders/dual_segformer_e5.py",
        "models/encoders/dual_segformer_e5.py.before_scr",
    ),
    (
        "train.py",
        "train.py.before_scr_identity",
    ),
]

for current_rel, backup_rel in files_to_check:
    current = ROOT / current_rel
    backup = ROOT / backup_rel

    print("\n" + "=" * 90)
    print("FILE:", current_rel)
    print("=" * 90)

    if not current.exists():
        print("Current file not found.")
        continue

    if not backup.exists():
        print("Backup not found:", backup)
        print("Showing current SCR-related lines instead.")

        for line_no, line in enumerate(
            current.read_text().splitlines(), start=1
        ):
            if any(term in line.lower() for term in [
                "mit_b2_scr",
                "spatialreliabilityconditionedfusion",
                "scr_init_path",
                "counterfactual",
                "optimizer audit",
            ]):
                print(f"{line_no:4}: {line}")
        continue

    old_lines = backup.read_text().splitlines()
    new_lines = current.read_text().splitlines()

    diff = difflib.unified_diff(
        old_lines,
        new_lines,
        fromfile=f"BACKUP/{backup_rel}",
        tofile=f"CURRENT/{current_rel}",
        lineterm="",
    )

    print("\n".join(diff) or "No differences found.")

print("\n" + "=" * 90)
print("NEW SCR FUSION MODULE")
print("=" * 90)

scr_module = ROOT / "models/net_utils_scr.py"

if scr_module.exists():
    print(scr_module.read_text())
else:
    print("SCR fusion module not found.")



FILE: config.py
--- BACKUP/config.py.before_scr_identity
+++ CURRENT/config.py
@@ -78,7 +78,7 @@
 # NETWORK CONFIGURATION
 # ============================================================
 # Network Configuration
-C.backbone = 'mit_b2_e5'
+C.backbone = 'mit_b2_scr'
 
 C.pretrained_model = '/home/cse-sdpl/Downloads/CMX_Original/pretrained/segformer/mit_b2.pth'
 

FILE: models/builder.py
--- BACKUP/models/builder.py.before_scr_identity
+++ CURRENT/models/builder.py
@@ -41,6 +41,10 @@
         elif cfg.backbone == 'mit_b2_e4':
             logger.info('Using E4: SegFormer-B2 with reliability-gated rectification')
             from .encoders.dual_segformer_e4 import mit_b2 as backbone
+            self.backbone = backbone(norm_fuse=norm_layer)
+        elif cfg.backbone == 'mit_b2_scr':
+            logger.info('Using SCR-CMX: Spatial Reliability-Conditioned Fusion')
+            from .encoders.dual_segformer_e5 import mit_b2 as backbone
             self.backbone = backbone(norm_fuse=norm_

In [22]:

from pathlib import Path
import shutil
import re
import os

ROOT = Path("/home/cse-sdpl/Downloads/CMX_SCR")
TEST_ROOT = ROOT / "_scr_smoke_test"

assert (ROOT / "train.py").is_file(), "train.py not found"
assert (ROOT / "config.py").is_file(), "config.py not found"
assert (
    ROOT / "log_TDD_mit_b2_e5/checkpoint/epoch-100.pth"
).is_file(), "E5 epoch-100 checkpoint not found"

if TEST_ROOT.exists():
    raise RuntimeError(
        f"Temporary test workspace already exists: {TEST_ROOT}\n"
        "Inspect it before removing or reusing it."
    )

TEST_ROOT.mkdir(parents=True)

# Copy the files that control the test.
shutil.copy2(ROOT / "train.py", TEST_ROOT / "train.py")
config_text = (ROOT / "config.py").read_text()

# Restrict the test to 1 epoch and 20 iterations.
# Only change assignments that match these exact setting names.
config_text, epochs_changed = re.subn(
    r"(?m)^(\s*C\.nepochs\s*=\s*)\d+",
    r"\g<1>1",
    config_text,
    count=1,
)
config_text, iterations_changed = re.subn(
    r"(?m)^(\s*C\.niters_per_epoch\s*=\s*)\d+",
    r"\g<1>20",
    config_text,
    count=1,
)

if epochs_changed != 1 or iterations_changed != 1:
    shutil.rmtree(TEST_ROOT)
    raise RuntimeError(
        "Could not safely patch nepochs/niters_per_epoch in config.py. "
        "No training was started. Inspect the exact config assignments first."
    )

(TEST_ROOT / "config.py").write_text(config_text)

# Link code, dataset, and pretrained assets to the original workspace.
# The temporary workspace has its own config.py and train.py.
for item in ROOT.iterdir():
    if item.name in {"config.py", "train.py", "_scr_smoke_test"}:
        continue
    if item.name.startswith("log_TDD_mit_b2_scr"):
        continue

    target = TEST_ROOT / item.name
    if not target.exists():
        target.symlink_to(item, target_is_directory=item.is_dir())

# Verify the temporary configuration.
test_config = (TEST_ROOT / "config.py").read_text()
assert re.search(r"(?m)^\s*C\.nepochs\s*=\s*1\s*$", test_config)
assert re.search(r"(?m)^\s*C\.niters_per_epoch\s*=\s*20\s*$", test_config)

print("SCR smoke-test workspace prepared.")
print("Temporary workspace:", TEST_ROOT)
print("Configured epochs: 1")
print("Configured iterations per epoch: 20")
print("Original config.py unchanged:", True)
print("Original SCR checkpoint directory untouched:", True)
print("No training started yet.")


RuntimeError: Could not safely patch nepochs/niters_per_epoch in config.py. No training was started. Inspect the exact config assignments first.

In [23]:

from pathlib import Path
import re

ROOT = Path("/home/cse-sdpl/Downloads/CMX_SCR")
CONFIG = ROOT / "config.py"

assert CONFIG.is_file(), f"Config not found: {CONFIG}"

lines = CONFIG.read_text().splitlines()

print("=== Matching training configuration lines ===")
for number, line in enumerate(lines, start=1):
    if re.search(
        r"nepochs|niters_per_epoch|batch_size|learning_rate|lr\s*=|backbone\s*=",
        line,
        re.IGNORECASE,
    ):
        print(f"{number:4}: {line}")

print("\n=== Exact assignment checks ===")
for name in ("nepochs", "niters_per_epoch"):
    matches = [
        (number, line)
        for number, line in enumerate(lines, start=1)
        if name in line and not line.lstrip().startswith("#")
    ]
    print(f"\n{name}:")
    if matches:
        for number, line in matches:
            print(f"  Line {number}: {line}")
    else:
        print("  No active matching line found.")

print("\nInspection only: no files modified; no training started.")


=== Matching training configuration lines ===
  81: C.backbone = 'mit_b2_scr'
  91: C.lr = 6e-5
  98: C.batch_size = 2
  99: C.nepochs = 100
 100: C.niters_per_epoch = C.num_train_imgs // C.batch_size + 1
 168:     print('Batch size:', C.batch_size)
 169:     print('Epochs:', C.nepochs)

=== Exact assignment checks ===

nepochs:
  Line 99: C.nepochs = 100
  Line 169:     print('Epochs:', C.nepochs)

niters_per_epoch:
  Line 100: C.niters_per_epoch = C.num_train_imgs // C.batch_size + 1

Inspection only: no files modified; no training started.


In [24]:

from pathlib import Path
import shutil
import re

ROOT = Path("/home/cse-sdpl/Downloads/CMX_SCR")
TEST_ROOT = ROOT / "_scr_smoke_test"

assert (ROOT / "train.py").is_file()
assert (ROOT / "config.py").is_file()

if TEST_ROOT.exists():
    raise RuntimeError(
        f"{TEST_ROOT} already exists. Inspect it before proceeding; "
        "this cell will not delete or overwrite it."
    )

config_text = (ROOT / "config.py").read_text()

# Patch only the temporary config, leaving the original unchanged.
config_text, epochs_changed = re.subn(
    r"(?m)^(\s*C\.nepochs\s*=\s*)\d+\s*$",
    r"\g<1>1",
    config_text,
    count=1,
)

config_text, iterations_changed = re.subn(
    r"(?m)^(\s*C\.niters_per_epoch\s*=\s*).*$",
    r"\g<1>20",
    config_text,
    count=1,
)

if epochs_changed != 1 or iterations_changed != 1:
    raise RuntimeError(
        f"Safe patch failed: epochs={epochs_changed}, "
        f"iterations={iterations_changed}. No files copied."
    )

# Copy source files and directories, excluding active SCR training outputs.
TEST_ROOT.mkdir(parents=True)

try:
    for item in ROOT.iterdir():
        if item.name in {"_scr_smoke_test", "config.py", "train.py"}:
            continue
        if item.name.startswith("log_TDD_mit_b2_scr"):
            continue

        destination = TEST_ROOT / item.name
        if item.is_dir():
            shutil.copytree(item, destination, symlinks=True)
        elif item.is_file():
            shutil.copy2(item, destination)

    shutil.copy2(ROOT / "train.py", TEST_ROOT / "train.py")
    (TEST_ROOT / "config.py").write_text(config_text)

    # Verify temporary settings.
    check = (TEST_ROOT / "config.py").read_text()
    assert re.search(r"(?m)^C\.nepochs\s*=\s*1\s*$", check)
    assert re.search(r"(?m)^C\.niters_per_epoch\s*=\s*20\s*$", check)
    assert "C.backbone = 'mit_b2_scr'" in check

except Exception:
    print("Preparation failed. Inspect the temporary directory before retrying.")
    raise

print("Smoke-test workspace prepared:", TEST_ROOT)
print("Temporary epochs: 1")
print("Temporary iterations per epoch: 20")
print("Original config.py unchanged.")
print("No training started.")


Smoke-test workspace prepared: /home/cse-sdpl/Downloads/CMX_SCR/_scr_smoke_test
Temporary epochs: 1
Temporary iterations per epoch: 20
Original config.py unchanged.
No training started.


In [28]:

from pathlib import Path
import subprocess
import re

ROOT = Path("/home/cse-sdpl/Downloads/CMX_SCR")
TEST_ROOT = ROOT / "_scr_smoke_test"

checks = {
    "Temporary train.py": TEST_ROOT / "train.py",
    "Temporary config.py": TEST_ROOT / "config.py",
    "SCR fusion module": TEST_ROOT / "models/net_utils_scr.py",
    "SCR encoder": TEST_ROOT / "models/encoders/dual_segformer_e5.py",
    "Dataset directory": TEST_ROOT / "datasets/TDD",
    "E5 checkpoint": TEST_ROOT / "log_TDD_mit_b2_e5/checkpoint/epoch-100.pth",
    "Pretrained MiT-B2 weights": TEST_ROOT / "pretrained/segformer/mit_b2.pth",
}

print("=== Workspace checks ===")
for label, path in checks.items():
    print(f"{label}: {'FOUND' if path.exists() else 'MISSING'}")
    if label == "E5 checkpoint" and path.is_file():
        print(f"  Size: {path.stat().st_size / (1024**3):.2f} GiB")

cfg = (TEST_ROOT / "config.py").read_text()
print("\n=== Temporary configuration ===")
for line in cfg.splitlines():
    if re.search(r"backbone\s*=|nepochs\s*=|niters_per_epoch\s*=|batch_size\s*=|lr\s*=", line):
        print(line)

print("\n=== GPU status (read-only) ===")
try:
    result = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=name,memory.used,memory.total,utilization.gpu",
            "--format=csv"
        ],
        capture_output=True,
        text=True,
        timeout=10,
        check=True,
    )
    print(result.stdout)
except Exception as exc:
    print("Could not read GPU status:", exc)

print("Verification complete. No training started.")


=== Workspace checks ===
Temporary train.py: FOUND
Temporary config.py: FOUND
SCR fusion module: FOUND
SCR encoder: FOUND
Dataset directory: MISSING
E5 checkpoint: FOUND
  Size: 0.75 GiB
Pretrained MiT-B2 weights: MISSING

=== Temporary configuration ===
C.backbone = 'mit_b2_scr'
C.lr = 6e-5
C.batch_size = 2
C.nepochs = 1
C.niters_per_epoch = 20

=== GPU status (read-only) ===
name, memory.used [MiB], memory.total [MiB], utilization.gpu [%]
NVIDIA RTX 2000 Ada Generation, 8557 MiB, 16380 MiB, 93 %

Verification complete. No training started.


In [29]:

from pathlib import Path

ROOT = Path("/home/cse-sdpl/Downloads/CMX_SCR")
TEST_ROOT = ROOT / "_scr_smoke_test"

candidates = [
    Path("/home/cse-sdpl/Downloads/CMX_Original/datasets/TDD"),
    Path("/home/cse-sdpl/Downloads/CMX_SCR/datasets/TDD"),
    Path("/home/cse-sdpl/Downloads/CMX_E5/datasets/TDD"),
    Path("/home/cse-sdpl/Downloads/CMX_Original/pretrained/segformer/mit_b2.pth"),
    Path("/home/cse-sdpl/Downloads/CMX_SCR/pretrained/segformer/mit_b2.pth"),
    Path("/home/cse-sdpl/Downloads/CMX_E5/pretrained/segformer/mit_b2.pth"),
]

print("=== Candidate paths ===")
for path in candidates:
    if path.exists():
        kind = "directory" if path.is_dir() else "file"
        print(f"FOUND ({kind}): {path}")
    else:
        print(f"not found: {path}")

print("\n=== Original project config paths ===")
original_config = ROOT / "config.py"
text = original_config.read_text()

for line in text.splitlines():
    if any(word in line.lower() for word in ("dataset_path", "data_path", "pretrained", "root_dir")):
        print(line)

print("\nRead-only inspection complete. No files changed.")


=== Candidate paths ===
FOUND (directory): /home/cse-sdpl/Downloads/CMX_Original/datasets/TDD
not found: /home/cse-sdpl/Downloads/CMX_SCR/datasets/TDD
not found: /home/cse-sdpl/Downloads/CMX_E5/datasets/TDD
FOUND (file): /home/cse-sdpl/Downloads/CMX_Original/pretrained/segformer/mit_b2.pth
not found: /home/cse-sdpl/Downloads/CMX_SCR/pretrained/segformer/mit_b2.pth
not found: /home/cse-sdpl/Downloads/CMX_E5/pretrained/segformer/mit_b2.pth

=== Original project config paths ===
C.root_dir = os.path.abspath(os.getcwd())
C.dataset_path = '/home/cse-sdpl/Downloads/CMX_Original/datasets/TDD'
C.rgb_root_folder = osp.join(C.dataset_path, 'RGB')
C.gt_root_folder = osp.join(C.dataset_path, 'Label')
C.x_root_folder = osp.join(C.dataset_path, 'Thermal')
C.train_source = osp.join(C.dataset_path, 'train.txt')
C.eval_source = osp.join(C.dataset_path, 'val.txt')
C.test_source = osp.join(C.dataset_path, 'test.txt')
C.pretrained_model = '/home/cse-sdpl/Downloads/CMX_Original/pretrained/segformer/mit_b2.

In [30]:

from pathlib import Path
import os

ROOT = Path("/home/cse-sdpl/Downloads/CMX_SCR")
TEST_ROOT = ROOT / "_scr_smoke_test"

DATASET = Path("/home/cse-sdpl/Downloads/CMX_Original/datasets/TDD")
PRETRAINED = Path(
    "/home/cse-sdpl/Downloads/CMX_Original/pretrained/segformer/mit_b2.pth"
)

assert TEST_ROOT.is_dir(), "Temporary workspace missing"
assert DATASET.is_dir(), f"Dataset missing: {DATASET}"
assert PRETRAINED.is_file(), f"Pretrained weights missing: {PRETRAINED}"

# Create symlinks only when the destination does not already exist.
links = {
    TEST_ROOT / "datasets": ROOT / "datasets",
    TEST_ROOT / "pretrained": ROOT / "pretrained",
}

for destination, source in links.items():
    if destination.exists() or destination.is_symlink():
        print(f"Already exists; leaving untouched: {destination}")
    else:
        destination.symlink_to(source, target_is_directory=True)
        print(f"Linked: {destination} -> {source}")

# Verify the paths that the temporary config actually uses.
config_text = (TEST_ROOT / "config.py").read_text()

print("\n=== Required paths ===")
print("Dataset exists:", DATASET.is_dir())
print("Train split exists:", (DATASET / "train.txt").is_file())
print("Validation split exists:", (DATASET / "val.txt").is_file())
print("Test split exists:", (DATASET / "test.txt").is_file())
print("RGB directory exists:", (DATASET / "RGB").is_dir())
print("Thermal directory exists:", (DATASET / "Thermal").is_dir())
print("Label directory exists:", (DATASET / "Label").is_dir())
print("Pretrained weights accessible:", PRETRAINED.is_file())
print("Temporary config unchanged from smoke-test settings:",
      "C.nepochs = 1" in config_text and "C.niters_per_epoch = 20" in config_text)

print("\nWorkspace prepared. No training started.")


Linked: /home/cse-sdpl/Downloads/CMX_SCR/_scr_smoke_test/datasets -> /home/cse-sdpl/Downloads/CMX_SCR/datasets
Already exists; leaving untouched: /home/cse-sdpl/Downloads/CMX_SCR/_scr_smoke_test/pretrained

=== Required paths ===
Dataset exists: True
Train split exists: True
Validation split exists: True
Test split exists: True
RGB directory exists: True
Thermal directory exists: True
Label directory exists: True
Pretrained weights accessible: True
Temporary config unchanged from smoke-test settings: True

Workspace prepared. No training started.
